In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest
import os

# ------------------------------
# Paths
# ------------------------------
file_path = r"C:\Users\INFOTEC\OneDrive\Bureau\pre_w4w5\Cross_Week_Results4\Week4_Final.xlsx"
output_folder = r"C:\Users\INFOTEC\OneDrive\Bureau\anomalie_w4"
os.makedirs(output_folder, exist_ok=True)

output_file = os.path.join(output_folder, "Toutes_les_anomaliesw4.xlsx")

# ------------------------------
# Utils
# ------------------------------
def to_numeric_safe(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = (
                pd.to_numeric(
                    df[col].astype(str).str.replace(",", "."),
                    errors="coerce"
                )
                .fillna(0)
            )
    return df

# ------------------------------
# Read Excel
# ------------------------------
sheets = pd.read_excel(file_path, sheet_name=None)

# ------------------------------
# Writer (FORCÉ)
# ------------------------------
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for pays, df in sheets.items():
        print(f"▶ Traitement : {pays}")

        cols = ["Real Inventory (Qty)", "Stock Value (€)"]
        df = to_numeric_safe(df, cols)

        if df.empty:
            df.to_excel(writer, sheet_name=pays[:31], index=False)
            continue

        model = IsolationForest(
            n_estimators=200,
            contamination=0.01,
            random_state=42
        )

        model.fit(df[cols])
        df["anomaly"] = model.predict(df[cols])
        df["anomaly_score"] = model.decision_function(df[cols])

        anomalies = df[df["anomaly"] == -1].copy()

        # Même si 0 anomalies → sheet créé
        anomalies.to_excel(writer, sheet_name=pays[:31], index=False)

        print(f"✔ {len(anomalies)} anomalies")

print("\n✅ FICHIER CRÉÉ AVEC SUCCÈS :")
print(output_file)


In [ ]:
# Statistiques descriptives sur les colonnes numériques uniquement
print(anomalies.describe())


In [ ]:
# Installer matplotlib et seaborn directement depuis Jupyter
!pip install matplotlib seaborn --upgrade


In [ ]:
!pip install --user matplotlib seaborn


In [ ]:
all_anomalies_list = []

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for pays, df in sheets.items():
        print(f"▶ Traitement : {pays}")

        cols = ["Real Inventory (Qty)", "Stock Value (€)"]
        df = to_numeric_safe(df, cols)

        if df.empty:
            df.to_excel(writer, sheet_name=pays[:31], index=False)
            continue

        model = IsolationForest(
            n_estimators=200,
            contamination=0.01,
            random_state=42
        )

        model.fit(df[cols])
        df["anomaly"] = model.predict(df[cols])
        df["anomaly_score"] = model.decision_function(df[cols])

        anomalies = df[df["anomaly"] == -1].copy()

        # Stocker dans la liste globale
        anomalies['pays'] = pays
        all_anomalies_list.append(anomalies)

        # Même si 0 anomalies → sheet créé
        anomalies.to_excel(writer, sheet_name=pays[:31], index=False)

        print(f"✔ {len(anomalies)} anomalies")

# Concaténer toutes les anomalies pour un DataFrame global
all_anomalies_df = pd.concat(all_anomalies_list, ignore_index=True)
print(f"\n✅ FICHIER CRÉÉ AVEC SUCCÈS : {output_file}")
print(f"Nombre total d'anomalies : {len(all_anomalies_df)}")

# ==============================
# Histogramme global
# ==============================
try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    if not all_anomalies_df.empty:
        plt.figure(figsize=(10,6))
        sns.histplot(all_anomalies_df['anomaly_score'], bins=30, kde=True)
        plt.title("Distribution globale des scores d'anomalie (tous les pays)")
        plt.xlabel("Score d'anomalie")
        plt.ylabel("Nombre d'occurrences")
        plt.show()
    else:
        print("Aucune anomalie détectée, plot ignoré.")

    # Top 10% anomalies
    top_n = max(1, int(len(all_anomalies_df) * 0.10))
    top_anomalies = all_anomalies_df.nsmallest(top_n, 'anomaly_score')
    print(f"\n=== Top 10% anomalies globales ===")
    print(top_anomalies)

except ImportError:
    print("Matplotlib ou Seaborn non installés, plots ignorés.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Colonnes numériques d'intérêt
numeric_cols = ["Real Inventory (Qty)", "Stock Value (€)"]

# Vérifier que les colonnes existent
numeric_cols = [c for c in numeric_cols if c in all_anomalies_df.columns]

for col in numeric_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=all_anomalies_df[col])
    plt.title(f"Boxplot des anomalies pour {col}")
    plt.xlabel(col)
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Choisir les 2 premières colonnes numériques
numeric_cols = ["Real Inventory (Qty)", "Stock Value (€)"]
numeric_cols = [c for c in numeric_cols if c in all_anomalies_df.columns]

if len(numeric_cols) >= 2:
    cols = numeric_cols[:2]
    plt.figure(figsize=(6,4))
    plt.scatter(
        all_anomalies_df[cols[0]], 
        all_anomalies_df[cols[1]], 
        color='red', alpha=0.6
    )
    plt.xlabel(cols[0])
    plt.ylabel(cols[1])
    plt.title("Scatter plot des anomalies")
    plt.show()

In [ ]:
# ==============================
# Statistiques et top anomalies avec graphes affichés dans Python
# ==============================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# -----------------------------
# Chemins
# -----------------------------
input_file = r"C:\Users\INFOTEC\OneDrive\Bureau\anomalie_w3\Toutes_les_anomaliesw4.xlsx"
report_folder = r"C:\Users\INFOTEC\OneDrive\Bureau\anomalie_w4\reportsw4"
os.makedirs(report_folder, exist_ok=True)

# Fichier Excel de sortie pour stats & top anomalies
stats_file = os.path.join(report_folder, "anomalies_statistics_topw3.xlsx")

# -----------------------------
# Lire toutes les sheets
# -----------------------------
sheets = pd.read_excel(input_file, sheet_name=None)

# -----------------------------
# Concaténation globale
# -----------------------------
all_data = []
for sheet_name, df in sheets.items():
    df = df.copy()
    df['pays'] = sheet_name
    all_data.append(df)

all_data_df = pd.concat(all_data, ignore_index=True)

# -----------------------------
# Colonnes numériques
# -----------------------------
numeric_cols = all_data_df.select_dtypes(include=['float64', 'int64']).columns

# -----------------------------
# 1️⃣ Statistiques descriptives & top anomalies dans Excel
# -----------------------------
with pd.ExcelWriter(stats_file) as writer:
    # Statistiques descriptives
    for col in numeric_cols:
        if col in all_data_df.columns:
            all_data_df[col].describe().to_frame().to_excel(writer, sheet_name=f"Stats_{col[:25]}")
    
    # Top anomalies globales (ex: Stock Value élevé)
    if 'Stock Value (€)' in all_data_df.columns:
        top_percent = 0.10
        top_n = max(1, int(len(all_data_df) * top_percent))
        top_anomalies = all_data_df.sort_values('Stock Value (€)', ascending=False).head(top_n)
        top_anomalies.to_excel(writer, sheet_name="Top_Anomalies", index=False)

print(f"✅ Statistiques et top anomalies sauvegardées dans : {stats_file}")

# -----------------------------
# 2️⃣ Graphiques affichés directement dans Python
# -----------------------------
for col in numeric_cols:
    if col in all_data_df.columns:
        plt.figure(figsize=(10,4))
        sns.histplot(all_data_df[col].dropna(), kde=True)
        plt.title(f"Histogramme : {col}")
        plt.show()

        plt.figure(figsize=(6,4))
        sns.boxplot(x=all_data_df[col].dropna())
        plt.title(f"Boxplot : {col}")
        plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Charger le fichier Excel
# -----------------------------
file = r"C:\Users\INFOTEC\OneDrive\Bureau\pre_w4w5\Cross_Week_Results4\Week4_Final.xlsx"
sheets = pd.read_excel(file, sheet_name=None)
print("Feuilles trouvées :", list(sheets.keys()))

# -----------------------------
# Boucle sur chaque pays (chaque feuille)
# -----------------------------
for pays, df in sheets.items():
    print(f"\n▶ Traitement : {pays}")

    # Colonnes importantes
    target_col = "Weekly Target"
    usage_col = "Weekly Usage (Qty)"

    # Vérifier que les colonnes existent
    if target_col not in df.columns or usage_col not in df.columns:
        print(f"⚠️ Colonnes manquantes pour {pays}: {[c for c in [target_col, usage_col] if c not in df.columns]}")
        continue

    # Conversion en numérique
    df[target_col] = pd.to_numeric(df[target_col].astype(str).str.replace(',', '.').str.strip(), errors='coerce').fillna(0)
    df[usage_col] = pd.to_numeric(df[usage_col].astype(str).str.replace(',', '.').str.strip(), errors='coerce').fillna(0)

    # -----------------------------
    # Graphique avec deux courbes
    # -----------------------------
    plt.figure(figsize=(8,5))
    plt.plot(df.index, df[target_col], marker='o', linestyle='-', color='blue', label='Weekly Target')
    plt.plot(df.index, df[usage_col], marker='o', linestyle='--', color='orange', label='Weekly Usage')

    plt.title(f"Weekly Target vs Weekly Usage – {pays}")
    plt.xlabel("Index")
    plt.ylabel("Quantity")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os

# =================================================
# PATHS
# =================================================
input_file = r"C:\Users\INFOTEC\OneDrive\Bureau\pre_w4w5\Cross_Week_Results4\Week4_Final.xlsx"
output_dir = r"C:\Users\INFOTEC\OneDrive\Bureau\anomalie_W4\anomalie_metier\\"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, "pays_files"), exist_ok=True)

output_file_all = os.path.join(output_dir, "Week4_With_Anomalies_Metier.xlsx")
output_file_anomaly = os.path.join(output_dir, "Anomalies_Metier_Only_W4.xlsx")

# =================================================
# UTILS
# =================================================
def to_numeric_series(s):
    return (
        s.astype(str)
         .str.replace(",", ".", regex=False)
         .str.replace(" ", "", regex=False)
         .replace("nan", np.nan)
         .astype(float)
    )

# =================================================
# LOAD EXCEL
# =================================================
print("📂 Chargement du fichier Excel...")
xl = pd.ExcelFile(input_file)
sheet_names = xl.sheet_names
print(f"✅ {len(sheet_names)} sheets trouvés")

all_data = {}
only_anomalies = []
stats_by_country = []

# =================================================
# PROCESS EACH COUNTRY
# =================================================
for country in sheet_names:
    print(f"\n🔍 Traitement : {country}")
    df = pd.read_excel(xl, sheet_name=country)

    # Sécurité colonnes
    for col in ["Real Inventory (Qty)", "WU", "Weekly Target"]:
        if col not in df.columns:
            df[col] = np.nan

    # Nettoyage numérique
    df["Inventory"] = to_numeric_series(df["Real Inventory (Qty)"])
    df["WU"] = to_numeric_series(df["WU"])
    df["Target"] = to_numeric_series(df["Weekly Target"])

    # ==============================
    # DÉTECTIONS MÉTIER
    # ==============================
    df["Anom_Inventory_Negatif"] = (df["Inventory"] < 0).astype(int)
    df["Anom_Rupture_Stock"] = (df["Inventory"] == 0).astype(int)
    df["Anom_Rotation_Rapide"] = (df["WU"] < 0.5).astype(int)
    df["Anom_Sous_Stock"] = ((df["Inventory"] < df["Target"]) & (df["Inventory"] > 0)).astype(int)
    df["Anom_Surstock_Critique"] = (df["WU"] > 6).astype(int)

    df["Total_Anomalies"] = df[
        [
            "Anom_Inventory_Negatif",
            "Anom_Rupture_Stock",
            "Anom_Rotation_Rapide",
            "Anom_Sous_Stock",
            "Anom_Surstock_Critique",
        ]
    ].sum(axis=1)

    # ==============================
    # ANOMALIE PRINCIPALE
    # ==============================
    conditions = [
        df["Inventory"] < 0,
        df["Inventory"] == 0,
        df["WU"] < 0.5,
        (df["Inventory"] < df["Target"]) & (df["Inventory"] > 0),
        df["WU"] > 6
    ]

    choices = [
        "Inventory négatif",
        "Rupture de stock",
        "Rotation trop rapide",
        "Sous-stock",
        "Sur-stock critique"
    ]

    df["Anomalie_Metier"] = np.select(conditions, choices, default="Normal")

    # ==============================
    # STATS
    # ==============================
    df_anom = df[df["Total_Anomalies"] > 0].copy()

    stats_by_country.append({
        "Pays": country,
        "Total_Lignes": len(df),
        "Lignes_Anormales": len(df_anom),
        "Pourcentage_Anomalies": round(len(df_anom) / len(df) * 100, 2) if len(df) > 0 else 0
    })

    all_data[country] = df
    only_anomalies.append(df_anom)

    # Fichier par pays
    df.to_excel(
        os.path.join(output_dir, "pays_files", f"{country}_Anomalies.xlsx"),
        index=False
    )

    print(f"   ✔ {len(df_anom)} anomalies détectées")

# =================================================
# EXPORT GLOBAL
# =================================================
stats_df = pd.DataFrame(stats_by_country)

print("\n💾 Création des fichiers Excel finaux...")

with pd.ExcelWriter(output_file_all, engine="openpyxl") as writer:
    stats_df.to_excel(writer, sheet_name="RESUME", index=False)
    for country, df in all_data.items():
        df.to_excel(writer, sheet_name=country[:31], index=False)

with pd.ExcelWriter(output_file_anomaly, engine="openpyxl") as writer:
    stats_df.to_excel(writer, sheet_name="RESUME", index=False)
    for i, df_anom in enumerate(only_anomalies):
        if not df_anom.empty:
            df_anom.to_excel(writer, sheet_name=f"Pays_{i+1}", index=False)

# =================================================
# FINAL
# =================================================
print("\n" + "=" * 70)
print("✨ TRAITEMENT TERMINÉ AVEC SUCCÈS")
print("=" * 70)
print(f"📁 {output_file_all}")
print(f"📁 {output_file_anomaly}")
print(f"📂 Dossier pays_files/ créé")
print("=" * 70)